In [6]:
from fpl_project.scripts import config
from fpl_project.scripts.data_utils.features_config import FeaturesConfig
from fpl_project.scripts.model_utils.data import FPLDataPipe, train_test_split
from fpl_project.scripts.model_utils.trainer import set_seed
from fpl_project.scripts.model_utils.selector import ModelSelector, initialize_device

from typing import NamedTuple
from dataclasses import dataclass
from typing import Any, Dict
from fpl_project.scripts.model_utils.models import FPLLoss

In [7]:
device = initialize_device()

In [8]:
import pandas as pd
import scipy.stats as stats

pd.set_option('display.max_columns', None)

from sklearn.ensemble import RandomForestRegressor
import xgboost
import catboost

import torch

import numpy as np
import seaborn as sns
sns.set_style("whitegrid")

import matplotlib.pyplot as plt


In [9]:
device = initialize_device()
device

'cuda'

In [10]:
set_seed(77)
data = pd.read_parquet("/home/onyxia/work/FPL/Data/fpl_data.parquet")

In [11]:
data.head()

,prob_under_2.5,prob_win_elo,match_value_efficiency,team,player_match_xg_expected,transfers_out,hype_vs_odds_ratio,transfers_balance,kickoff_time,transfers_trend,prob_draw,elo_opp,element,prob_lose,elo,player_match_xa_expected,transfers_in,name,selected,value,code,gw,season,opponent_team,position,was_home,total_points,bonus_ema_3,bps_ema_3,ict_index_ema_3,threat_ema_3,creativity_ema_3,value_ema_3,transfers_in_ema_3,transfers_out_ema_3,transfers_balance_ema_3,transfers_trend_ema_3,goals_scored_ema_3,assists_ema_3,own_goals_ema_3,penalties_missed_ema_3,xg_ema_3,xa_ema_3,xg_per_90_ema_3,xa_per_90_ema_3,xg_involvements_per_90_ema_3,goals_conceded_ema_3,xg_conceded_ema_3,xg_conceded_per_90_ema_3,clean_sheets_ema_3,saves_ema_3,penalties_saved_ema_3,minutes_ema_3,yellow_cards_ema_3,red_cards_ema_3,team_h_score_ema_3,team_a_score_ema_3,xp_ema_3,prob_win_ema_3,prob_draw_ema_3,prob_lose_ema_3,prob_over_2.5_ema_3,elo_ema_3,elo_diff_ema_3,player_match_xg_expected_ema_3,player_match_xa_expected_ema_3,match_value_efficiency_ema_3,hype_vs_odds_ratio_ema_3,bonus_lagged_1,transfers_balance_lagged_1,transfers_trend_lagged_1,goals_scored_lagged_1,assists_lagged_1,own_goals_lagged_1,penalties_missed_lagged_1,xg_lagged_1,xa_lagged_1,xg_per_90_lagged_1,xa_per_90_lagged_1,xg_involvements_per_90_lagged_1,goals_conceded_lagged_1,xg_conceded_per_90_lagged_1,clean_sheets_lagged_1,penalties_saved_lagged_1,yellow_cards_lagged_1,red_cards_lagged_1,team_h_score_lagged_1,team_a_score_lagged_1,player_match_xg_expected_lagged_1,player_match_xa_expected_lagged_1,hype_vs_odds_ratio_lagged_1,bonus_lagged_2,bps_lagged_2,ict_index_lagged_2,influence_lagged_2,threat_lagged_2,creativity_lagged_2,value_lagged_2,transfers_in_lagged_2,transfers_out_lagged_2,transfers_balance_lagged_2,transfers_trend_lagged_2,goals_scored_lagged_2,assists_lagged_2,own_goals_lagged_2,penalties_missed_lagged_2,xg_lagged_2,xa_lagged_2,xg_per_90_lagged_2,xa_per_90_lagged_2,xg_involvements_lagged_2,xg_involvements_per_90_lagged_2,goals_conceded_lagged_2,xg_conceded_lagged_2,xg_conceded_per_90_lagged_2,clean_sheets_lagged_2,saves_lagged_2,penalties_saved_lagged_2,minutes_lagged_2,yellow_cards_lagged_2,red_cards_lagged_2,team_h_score_lagged_2,team_a_score_lagged_2,xp_lagged_2,prob_win_lagged_2,prob_draw_lagged_2,prob_lose_lagged_2,prob_over_2.5_lagged_2,elo_lagged_2,elo_diff_lagged_2,player_match_xg_expected_lagged_2,player_match_xa_expected_lagged_2,match_value_efficiency_lagged_2,hype_vs_odds_ratio_lagged_2,bonus_lagged_3,bps_lagged_3,ict_index_lagged_3,influence_lagged_3,threat_lagged_3,creativity_lagged_3,value_lagged_3,transfers_in_lagged_3,transfers_out_lagged_3,transfers_balance_lagged_3,transfers_trend_lagged_3,goals_scored_lagged_3,assists_lagged_3,own_goals_lagged_3,penalties_missed_lagged_3,xg_lagged_3,xa_lagged_3,xg_per_90_lagged_3,xa_per_90_lagged_3,xg_involvements_lagged_3,xg_involvements_per_90_lagged_3,goals_conceded_lagged_3,xg_conceded_lagged_3,xg_conceded_per_90_lagged_3,clean_sheets_lagged_3,saves_lagged_3,penalties_saved_lagged_3,minutes_lagged_3,yellow_cards_lagged_3,red_cards_lagged_3,team_h_score_lagged_3,team_a_score_lagged_3,xp_lagged_3,prob_win_lagged_3,prob_draw_lagged_3,prob_lose_lagged_3,prob_over_2.5_lagged_3,elo_diff_lagged_3,player_match_xg_expected_lagged_3,player_match_xa_expected_lagged_3,hype_vs_odds_ratio_lagged_3,bonus_lagged_5,bps_lagged_5,ict_index_lagged_5,influence_lagged_5,threat_lagged_5,creativity_lagged_5,value_lagged_5,transfers_in_lagged_5,transfers_out_lagged_5,transfers_balance_lagged_5,transfers_trend_lagged_5,goals_scored_lagged_5,assists_lagged_5,own_goals_lagged_5,penalties_missed_lagged_5,xg_lagged_5,xa_lagged_5,xg_per_90_lagged_5,xa_per_90_lagged_5,xg_involvements_lagged_5,xg_involvements_per_90_lagged_5,goals_conceded_lagged_5,xg_conceded_lagged_5,xg_conceded_per_90_lagged_5,clean_sheets_lagged_5,saves_lagged_5,penalties_saved_lagged_5,minutes_lagged_5,yellow_cards_lagged_5,red_cards_lagged_5,team_h_score_lagged_5,team_a_score_lagged_5,x

In [12]:
cols_map = FeaturesConfig()


In [13]:
ignored_num = {"name", "position", "element", "opponent_team", "gw", "code", "season", "kickoff_time", "team"}
ignored_cat = {"web_name", "player_id", "gw"}
data_cols_set = set(data.columns)

ema_lagg_cols = [c for c in data.columns if "ema" in c or "lagg" in c]
pre_game_filtered = [c for c in getattr(cols_map, "pre_game_cols") if c in data_cols_set and c not in ignored_num]
num_cols = list(dict.fromkeys(ema_lagg_cols + pre_game_filtered))
cat_cols = [c for c in getattr(cols_map, "static_cols") if c in data_cols_set and c not in ignored_cat]



fpl_data_pipe = FPLDataPipe(num_cols=num_cols, cat_cols=cat_cols)


In [14]:
class PipelineState(NamedTuple):
    data_splits: dict
    preprocessor: Any

@dataclass
class ModelConfig:
    model: Any
    param_grid: Dict[str, Any]

@dataclass
class ParamGrid:
    catboost: Dict[str, Any]
    xgboost: Dict[str, Any]
    rf: Dict[str, Any]

device = "cpu"

param_grid = ParamGrid(
    rf={
        "n_estimators": stats.randint(150, 500),
        "max_depth": stats.randint(4, 15),
        "min_samples_split": stats.randint(2, 11),
        "min_samples_leaf": stats.randint(1, 10),
        "max_features": stats.uniform(0.6, 0.4),
    },
    catboost={
        "iterations": stats.randint(150, 500),
        "depth": stats.randint(4, 7), 
        "learning_rate": stats.uniform(0.01, 0.05),
        "l2_leaf_reg": stats.uniform(1.0, 8.0),
        "bagging_temperature": stats.uniform(0.0, 5.0),
        "random_strength": stats.uniform(0.5, 5.0),
        "border_count": stats.randint(32, 255),
    },
    xgboost={
        "n_estimators": stats.randint(150, 500),
        "max_depth": stats.randint(3, 7),
        "learning_rate": stats.uniform(0.01, 0.05),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.6, 0.3),
        "min_child_weight": stats.randint(1, 10),
        "gamma": stats.uniform(0.0, 5.0),
        "reg_alpha": stats.uniform(0.0, 2.0),
        "reg_lambda": stats.uniform(0.5, 3.0),
    }
    
)

model_registry = {
    "rf": ModelConfig(
        model=RandomForestRegressor(
            random_state=77,
        ),
        param_grid=param_grid.rf
    ),
    "catboost": ModelConfig(
        model=catboost.CatBoostRegressor(
            random_state=77, 
            loss_function="RMSE", 
            task_type="GPU" if device=="cuda" else "CPU",
            verbose=0
        ),
        param_grid=param_grid.catboost
    ),
    "xgboost": ModelConfig(
        model=xgboost.XGBRegressor(
            random_state=77, 
            objective='reg:squarederror', 
            eval_metric='rmse', 
            device=device
        ),  
        param_grid=param_grid.xgboost
    )
    
}

In [15]:
data_splits = train_test_split(data)
pipeline_state = PipelineState(data_splits=data_splits,
                            preprocessor=fpl_data_pipe.preprocessor)

In [16]:
def calculate_scaled_bounds(df: pd.DataFrame, value_idx: int) -> tuple[float, float]:
    prices = df.iloc[:, value_idx].to_numpy()
    
    return float(np.quantile(prices, 0.50)), float(np.quantile(prices, 0.955))

In [17]:
def split_xy(pipeline_state: PipelineState):
    data_splits, preprocessor = pipeline_state.data_splits, pipeline_state.preprocessor
    X_train = preprocessor.fit_transform(data_splits["X_train"]).drop(columns=["date"])
    X_valid = preprocessor.transform(data_splits["X_valid"]).drop(columns=["date"])
    X_test  = preprocessor.transform(data_splits["X_test"]).drop(columns=["date"])

    y_train = data_splits["y_train"]
    y_valid = data_splits["y_valid"]
    y_test  = data_splits["y_test"]

    return (X_train, X_valid, X_test, y_train, y_valid, y_test)



In [18]:
def train_models(model_registry: dict, pipeline_state: PipelineState):
    data_splits = pipeline_state.data_splits

    results_list = []
    params = {}

    for name, cfg in model_registry.items():
        model = cfg.model
        grid = cfg.param_grid

        X_train = data_splits.get("X_train").copy()
        y_train = data_splits.get("y_train").copy()

        if name == "catboost":
            fpl_pipe_cb = FPLDataPipe(num_cols=num_cols, cat_cols=cat_cols, catboost_flag=True)
            X_train = fpl_pipe_cb.preprocessor.fit_transform(X_train)
            
            cat_features = [col for col in X_train.columns if col.startswith("col__")]
            X_train[cat_features] = X_train[cat_features].astype(str)
        else:
            fpl_pipe_xgb = FPLDataPipe(num_cols=num_cols, cat_cols=cat_cols, catboost_flag=False)
            X_train = fpl_pipe_xgb.preprocessor.fit_transform(X_train)
            cat_features = None

        val_col_name = [col for col in X_train.columns if "value" in col][0]
        value_idx = X_train.columns.get_loc(val_col_name)

        p_low, p_high = calculate_scaled_bounds(df=X_train, value_idx=value_idx)
        
        selector = ModelSelector(
            random_state=77, 
            p_low=p_low, 
            p_high=p_high, 
            value_idx=value_idx
        )
        
        

        print(f"Model: {name} | shape: {X_train.shape} | val_idx: {value_idx}", flush=True)
        
        df_results, best_models_map = selector.params_search(
            models=[model],
            models_names=[name],
            params_grid=[grid], 
            X_train=X_train,
            y_train=y_train,
            cv=5,
            n_iter=20,
            scoring="neg_mean_squared_error",
            cat_features=cat_features
        )

        results_list.append(df_results)
        params[name] = best_models_map[name]

    return pd.concat(results_list, axis=0), params

In [ ]:
results, params = train_models(
    model_registry=model_registry, 
    pipeline_state=pipeline_state)

Model: rf | shape: (56230, 210) | val_idx: 9
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
results

,data,name,cv_mean_neg_mean_squared_error,best_params,mse@budget,mse@mid,mse@premium,mae@budget,mae@mid,mae@premium,r2
0,train,catboost,-2.931952,"{'bagging_temperature': 0.4833623041642121, 'b...",1.484878,3.105585,7.368887,0.516448,0.947532,1.829929,0.503479
0,train,xgboost,-2.833394,"{'colsample_bytree': 0.7266514068700778, 'gamm...",1.102418,2.060681,3.330692,0.433689,0.771936,1.215694,0.675285
